In [1]:
# --- Célula 1: Setup ---
!pip install -q mlflow scikit-learn

import mlflow
import mlflow.sklearn
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

print(f"MLflow version: {mlflow.__version__}")
# (Passo 2: Experimento 1 - Modelo Baseline)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.5/857.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.

In [2]:
# --- Célula 2: Treinamento com Tracking ---
# Carregar dados (Diabetes dataset - prever progressão da doença)
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Iniciar experimento MLflow
mlflow.set_experiment("diabetes_prediction")

# Treinar Modelo 1 (Baseline)
with mlflow.start_run(run_name="RandomForest_Baseline"):
  # Hiperparâmetros
  n_estimators = 10
  max_depth = 5

# Logar parâmetros
mlflow.log_param("n_estimators", n_estimators)
mlflow.log_param("max_depth", max_depth)

# Treinar
model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
model.fit(X_train, y_train)

# Predição
y_pred = model.predict(X_test)

# Métricas
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Logar métricas
mlflow.log_metric("rmse", rmse)
mlflow.log_metric("r2", r2)

# Salvar modelo
mlflow.sklearn.log_model(model, "model")

print(f"Modelo Baseline - RMSE: {rmse:.2f}, R2: {r2:.4f}")
# (Passo 3: Experimento 2 - Modelo Otimizado)


2026/04/20 11:10:00 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/20 11:10:00 INFO mlflow.store.db.utils: Updating database tables
2026/04/20 11:10:03 INFO mlflow.tracking.fluent: Experiment with name 'diabetes_prediction' does not exist. Creating a new experiment.
2026/04/20 11:10:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 11:10:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Modelo Baseline - RMSE: 56.63, R2: 0.3946


In [5]:
# --- Célula 3: Modelo Melhorado ---
# Ensure any active MLflow run is ended before starting a new one
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name="RandomForest_Optimized"):
  # Hiperparâmetros melhores
  n_estimators = 100
  max_depth = 10

  mlflow.log_param("n_estimators", n_estimators)
  mlflow.log_param("max_depth", max_depth)

  model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  rmse = np.sqrt(mean_squared_error(y_test, y_pred))
  r2 = r2_score(y_test, y_pred)

  mlflow.log_metric("rmse", rmse)
  mlflow.log_metric("r2", r2)
  mlflow.sklearn.log_model(model, "model")

  print(f"Modelo Otimizado - RMSE: {rmse:.2f}, R2: {r2:.4f}")
# (Passo 4: Comparando Experimentos)


2026/04/20 11:14:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 11:14:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Modelo Otimizado - RMSE: 54.54, R2: 0.4386


In [8]:
# --- Célula 4: Buscar Experimentos ---
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("diabetes_prediction")

# Buscar todas as runs
runs = client.search_runs(experiment.experiment_id)

print("--- Comparação de Experimentos ---")
for run in runs:
    print(f"Run: {run.info.run_name}")
    rmse_value = run.data.metrics.get('rmse', 'N/A')
    r2_value = run.data.metrics.get('r2', 'N/A')

    if rmse_value != 'N/A':
        print(f" RMSE: {rmse_value:.2f}")
    else:
        print(f" RMSE: {rmse_value}")

    if r2_value != 'N/A':
        print(f" R2: {r2_value:.4f}")
    else:
        print(f" R2: {r2_value}")

    print(f" Parâmetros: {run.data.params}")
    print()
# (Passo 5: Carregando Modelo para Inferência)


--- Comparação de Experimentos ---
Run: RandomForest_Optimized
 RMSE: 54.54
 R2: 0.4386
 Parâmetros: {'n_estimators': '100', 'max_depth': '10'}

Run: worried-eel-225
 RMSE: 56.63
 R2: 0.3946
 Parâmetros: {'n_estimators': '10', 'max_depth': '5'}

Run: RandomForest_Baseline
 RMSE: N/A
 R2: N/A
 Parâmetros: {}



In [9]:
# --- Célula 5: Servindo o Modelo (Batch Inference) ---
# Pegar o melhor modelo (maior R2)
best_run = max(runs, key=lambda r: r.data.metrics.get('r2', 0))
model_uri = f"runs:/{best_run.info.run_id}/model"

# Carregar modelo
loaded_model = mlflow.sklearn.load_model(model_uri)

# Fazer predições em novos dados (simulando produção)
novos_dados = X_test[:5]
predicoes = loaded_model.predict(novos_dados)

print("--- Inferência em Produção ---")
print(f"Predições: {predicoes}")
print(f"Valores Reais: {y_test[:5]}")
# (Passo 6: Simulando uma API REST)


--- Inferência em Produção ---
Predições: [142.18570502 172.94658289 154.13778627 252.87840952 109.05481766]
Valores Reais: [219.  70. 202. 230. 111.]


In [10]:
# --- Célula 6: Mock de API ---
def predict_api(features):
  """
  Simula uma função de API que recebe features e retorna predição.
  Em produção, isso seria um endpoint FastAPI ou Flask.
  """
  prediction = loaded_model.predict([features])
  return {"prediction": float(prediction[0])}

# Testar a "API"
exemplo_request = X_test[0].tolist()
response = predict_api(exemplo_request)

print(f"Request (Features): {exemplo_request}")
print(f"Response (API): {response}")
# (Análise do Lab) Você acabou de implementar o ciclo completo de MLOps: 1. Tracking: Comparou 2 modelos. 2. Registry: Salvou o melhor modelo. 3. Serving: Carregou o modelo e fez inferência. 4. API Mock: Simulou um endpoint de produção.
# Em um ambiente real, você usaria: * MLflow Server (hospedado em EC2/Azure VM) para centralizar os experimentos. * Docker para empacotar o modelo. * Kubernetes para escalar a API.


Request (Features): [0.04534098333546186, -0.044641636506989144, -0.006205954135807083, -0.015998975220305175, 0.12501870313429186, 0.1251981011367534, 0.019186997017453092, 0.03430885887772673, 0.03243232415655107, -0.005219804415300423]
Response (API): {'prediction': 142.18570502058915}


### Conclusão do Laboratório de MLOps

Neste exercício, você percorreu as etapas fundamentais do ciclo de vida de um modelo de aprendizado de máquina:

1.  **Tracking (Rastreamento):** Usamos o MLflow para registrar diferentes versões do modelo (`Baseline` vs `Optimized`), comparando métricas como RMSE e R2 para decidir qual era o melhor.
2.  **Model Registry (Registro):** Salvamos o modelo treinado e seus artefatos, permitindo que ele seja recuperado de forma consistente através de um `run_id` ou `model_uri`.
3.  **Inference (Inferência):** Demonstramos como carregar o melhor modelo salvo para realizar previsões em novos dados (Batch Inference).
4.  **Serving (Serviço):** Criamos um mock de API para simular como esse modelo seria consumido por outras aplicações em um ambiente de produção.

Essas práticas garantem que o desenvolvimento de modelos seja **reproduzível, auditável e fácil de implantar**.